# POPPy — Phytotherapies Ontology Build Pipeline

End-to-end notebook used to build the POPPy phytotherapy ontology. It loads the
hand-curated scaffold (`poppystructure.rdf`), adds plants/compounds from COCONUT,
CMAUP, DrugCentral, ChEMBL and Dr. Duke's, integrates ChEBI, then enriches with
external names (PubChem/ChEBI), cross-references (UniChem), NP classes
(NPClassifier), and the COCONUT 2.0 occurrence set.

## Requirements
- `pip install -e ".[dev,chem]"` (rdflib, rdkit, pandas, requests, psycopg2, biopython, tqdm)
- Environment variables: `ROMANO_DB_PASSWORD` (PMACS opendata DB), `ENTREZ_EMAIL`
- Input files (not committed — see `data/SOURCES.md`): `poppystructure.rdf`,
  CMAUP v2.0 `.txt` files, `ChEMBL_drug_mechanisms.csv`, Dr. Duke's CSVs,
  ChEBI (`chebi_lite.owl`), `coconut_sdf_2d-06-2026.sdf`.

## How to run
Run top to bottom. Each stage builds on the in-memory graph `g` and saves an RDF
checkpoint, so you can also resume by loading the latest saved file into `g`.
Large outputs are hosted on Box, not in the repo.

> Run **Stage 1 (Setup)** first — it defines the namespace, all predicate URIRefs,
> `PHYTOCHEM_CLASSES`, and every helper (`get_sql_conn`, `make_uri`, `classify_smiles`,
> `load_graph`, `save_graph`, …).

## 1. Setup

In [ ]:
# ---- Imports ---------------------------------------------------------------
import os, re, json, time, copy, pickle, hashlib
from pathlib import Path
from collections import defaultdict, deque
from typing import Iterable, Optional

import pandas as pd
import requests
import psycopg2
from tqdm import tqdm

from rdkit import Chem
from rdkit.Chem import Descriptors, MACCSkeys, rdMolDescriptors
try:
    from rdkit.Chem import inchi as RDInchi
    HAS_INCHI = True
except Exception:
    HAS_INCHI = False

from rdflib import Graph, Namespace, RDF, RDFS, OWL, Literal, URIRef, BNode
from rdflib.namespace import XSD, SKOS

from Bio import Entrez


# ---- Credentials (use environment variables) -------------------------------
# Set these in a .env or your shell:
#   export ROMANO_DB_PASSWORD=...
#   export ENTREZ_EMAIL=you@example.com
PMACS_HOST = os.environ.get("PMACS_HOST", "romanodb1.pmacs.upenn.edu")
PMACS_USER = os.environ.get("PMACS_USER", "ohewryk")
PMACS_PASS = os.environ.get("ROMANO_DB_PASSWORD")
DRUGCENTRAL_HOST = "unmtid-dbs.net"
DRUGCENTRAL_USER = "drugman"
DRUGCENTRAL_PASS = "dosage"   # public read-only DrugCentral mirror
ENTREZ_EMAIL = os.environ.get("ENTREZ_EMAIL", "ohewryk@sas.upenn.edu")
Entrez.email = ENTREZ_EMAIL

if not PMACS_PASS:
    print("[WARN] ROMANO_DB_PASSWORD env var is not set. SQL steps will fail.")


# ---- Namespace -------------------------------------------------------------
NS = Namespace("http://www.semanticweb.org/orestah/ontologies/2024/9/phytotherapies#")

# Object/data properties used across the pipeline
isDerivedFrom        = NS.isDerivedFrom
hasCompound          = NS.hasCompound
hasMolecularWeight   = NS.hasMolecularWeight
hasMolecularFormula  = NS.hasMolecularFormula
hasCommonName        = NS.hasCommonName
hasIUPACName         = NS.hasIUPACName
hasSMILES            = NS.hasSMILES
hasCanonicalSMILES   = NS.hasCanonicalSMILES
hasMACCs             = NS.hasMACCs
hasInChIKey          = NS.hasInChIKey
hasATC               = NS.hasATC
hasDOI               = NS.hasDOI
hasPaper             = NS.hasPaper
hasClinicalStudy     = NS.hasClinicalStudy
hasEvidenceText      = NS.hasEvidenceText
hasNCTId             = NS.hasNCTId
hasTherapeuticEffect = NS.hasTherapeuticEffect
targetsPathway       = NS.targetsPathway
hasValue             = NS.hasValue
hasGenus             = NS.hasGenus
hasSpecies           = NS.hasSpecies
hasTaxon             = NS.hasTaxon

# Phytochemical subclasses (must already exist as owl:Class in the base ontology)
PHYTOCHEM_CLASSES = {
    "Carotenoid":     NS.Carotenoid,
    "DietaryFiber":   NS.DietaryFiber,
    "Isoprenoid":     NS.Isoprenoid,
    "Phytosterol":    NS.Phytosterol,
    "Polyphenol":     NS.Polyphenol,
    "Polysaccharide": NS.Polysaccharide,
    "Saponin":        NS.Saponin,
    "Unknown":        NS.Unknown,
}


# ---- Helpers ---------------------------------------------------------------
def get_sql_conn(dbname="opendata", host=None, user=None, password=None, port=5432):
    """Open a PostgreSQL connection. Defaults to PMACS opendata."""
    return psycopg2.connect(
        host=host or PMACS_HOST,
        port=port,
        dbname=dbname,
        user=user or PMACS_USER,
        password=password or PMACS_PASS,
        sslmode="prefer",
        gssencmode="disable",
    )


def get_drugcentral_conn():
    return psycopg2.connect(
        host=DRUGCENTRAL_HOST, port=5433, dbname="drugcentral",
        user=DRUGCENTRAL_USER, password=DRUGCENTRAL_PASS,
    )


def sanitize_for_uri(text: str) -> str:
    """Make a safe fragment for IRIs: spaces/odd chars → _, collapse, trim, cap."""
    s = re.sub(r"\s+", "_", str(text).strip())
    s = re.sub(r"[^\w\-\.]", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s[:200] or "unknown"


def make_uri(kind: str, value) -> URIRef:
    return URIRef(f"{NS}{kind}_{sanitize_for_uri(value)}")


def canon_smiles(smiles: str) -> str:
    s = (smiles or "").strip()
    if not s:
        return ""
    m = Chem.MolFromSmiles(s)
    return Chem.MolToSmiles(m, canonical=True) if m else ""


def normalize_doi(doi: str) -> str:
    d = str(doi).strip()
    d = re.sub(r'^https?://(dx\.)?doi\.org/', '', d, flags=re.I)
    d = re.sub(r'^(doi|dois)\s*:?\s*', '', d, flags=re.I)
    return d.rstrip(").,;]")


def find_class_by_label_or_local(g: Graph, names):
    """Find an owl:Class by rdfs:label OR by local name under the default NS."""
    names = list(names)
    for n in names:
        for s in g.subjects(RDFS.label, Literal(n)):
            if (s, RDF.type, OWL.Class) in g:
                return s
    for s in g.subjects(RDF.type, OWL.Class):
        local = str(s).rsplit("#", 1)[-1].rsplit("/", 1)[-1]
        if local in names:
            return s
    return None


def is_subclass_of(g: Graph, child, root) -> bool:
    if not child or not root or child == root:
        return child == root
    seen = {child}; q = deque([child])
    while q:
        cur = q.popleft()
        for sup in g.objects(cur, RDFS.subClassOf):
            if sup == root:
                return True
            if sup not in seen:
                seen.add(sup); q.append(sup)
    return False


def subclass_closure(g: Graph, root) -> set:
    out = set([root]); q = deque([root])
    while q:
        c = q.popleft()
        for sub in g.subjects(RDFS.subClassOf, c):
            if (sub, RDF.type, OWL.Class) in g and sub not in out:
                out.add(sub); q.append(sub)
    return out


def classify_smiles(smiles: str):
    """Heuristic phytochemical classifier. Returns list of labels."""
    mol = Chem.MolFromSmiles(smiles or "")
    if mol is None:
        return ["Unknown"]
    classes = []
    mw = Descriptors.MolWt(mol)
    num_rings = Descriptors.RingCount(mol)
    num_oh = len(mol.GetSubstructMatches(Chem.MolFromSmarts("[OH]")))
    num_c = sum(1 for a in mol.GetAtoms() if a.GetSymbol() == "C")
    num_o = sum(1 for a in mol.GetAtoms() if a.GetSymbol() == "O")
    num_double_bonds = len(mol.GetSubstructMatches(Chem.MolFromSmarts("C=C")))
    if num_rings >= 2 and num_oh >= 2: classes.append("Polyphenol")
    if num_c >= 30 and num_double_bonds >= 5: classes.append("Carotenoid")
    if num_rings >= 4 and num_oh >= 1: classes.append("Phytosterol")
    if num_c >= 40 and num_o >= 5: classes.append("Saponin")
    if mw > 500 and num_o > 10: classes.append("Polysaccharide")
    if num_c > 20 and num_o > 15: classes.append("DietaryFiber")
    if "C=C" in (smiles or "") and num_c % 5 == 0: classes.append("Isoprenoid")
    return classes or ["Unknown"]


def set_mw_range_union(g: Graph):
    """Allow hasMolecularWeight to range over (xsd:float ⊔ xsd:double)."""
    if (hasMolecularWeight, RDF.type, OWL.DatatypeProperty) not in g:
        g.add((hasMolecularWeight, RDF.type, OWL.DatatypeProperty))
        g.add((hasMolecularWeight, RDFS.label, Literal("hasMolecularWeight")))
    for o in list(g.objects(hasMolecularWeight, RDFS.range)):
        g.remove((hasMolecularWeight, RDFS.range, o))
    u, a, b = BNode(), BNode(), BNode()
    g.add((u, RDF.type, OWL.DataRange))
    g.add((u, OWL.unionOf, a))
    g.add((a, RDF.first, XSD.float));  g.add((a, RDF.rest, b))
    g.add((b, RDF.first, XSD.double)); g.add((b, RDF.rest, RDF.nil))
    g.add((hasMolecularWeight, RDFS.range, u))


def load_graph(path: str) -> Graph:
    g = Graph()
    fmt = "turtle" if path.lower().endswith((".ttl", ".turtle")) else "xml"
    g.parse(path, format=fmt)
    return g


def save_graph(g: Graph, path: str):
    fmt = "turtle" if path.lower().endswith((".ttl", ".turtle")) else "xml"
    g.serialize(path, format=fmt)
    print(f"Saved → {path} ({len(g):,} triples)")


## 2. Base ontology + plants from COCONUT

In [ ]:
# Load the base ontology and ensure the Plant subclass exists under PlantTaxonomy
g = load_graph("poppystructure.rdf")

if (NS.Plant, RDF.type, OWL.Class) not in g:
    g.add((NS.Plant, RDF.type, OWL.Class))
    g.add((NS.Plant, RDFS.subClassOf, NS.PlantTaxonomy))
    g.add((NS.Plant, RDFS.label, Literal("Plant")))

print("Plant class ready in ontology.")


In [ ]:
# Validate organisms via NCBI Taxonomy (primary) and POWO (fallback), then add as Plant individuals
def is_plantae(scientific_name: str) -> bool:
    try:
        search = Entrez.esearch(db="taxonomy", term=scientific_name)
        ids = Entrez.read(search)["IdList"]; search.close()
        if not ids:
            return False
        fetch = Entrez.efetch(db="taxonomy", id=ids[0], retmode="xml")
        rec = Entrez.read(fetch)[0]; fetch.close()
        lineage = rec.get("Lineage", "").lower()
        return "plantae" in lineage or "viridiplantae" in lineage
    except Exception:
        return False


def is_powo_plant(scientific_name: str) -> bool:
    try:
        r = requests.get(
            f"https://powo.science.kew.org/api/2/search?q={scientific_name.replace(' ', '%20')}",
            timeout=10,
        )
        return r.status_code == 200 and bool(r.json().get("results"))
    except Exception:
        return False


def normalize_scientific_name(name: str) -> str:
    parts = (name or "").strip().split()
    if len(parts) >= 2:
        return f"{parts[0].capitalize()} {parts[1].lower()}"
    return parts[0].capitalize() if parts else ""


# Cache results between runs
try:
    with open("plant_validation_cache.pkl", "rb") as f:
        validation_cache = pickle.load(f)
except FileNotFoundError:
    validation_cache = {}

with get_sql_conn() as conn, conn.cursor() as cur:
    cur.execute("SELECT id, name FROM coconut.organisms")
    organisms = cur.fetchall()

added = 0
for org_id, name in tqdm(organisms):
    if not name:
        continue
    clean = normalize_scientific_name(name)
    if clean not in validation_cache:
        is_plant = is_plantae(clean) or is_powo_plant(clean)
        validation_cache[clean] = is_plant
        if len(validation_cache) % 100 == 0:
            with open("plant_validation_cache.pkl", "wb") as f:
                pickle.dump(validation_cache, f)
        time.sleep(0.2)
    if validation_cache[clean]:
        uri = NS[f"Organism_{org_id}"]
        g.add((uri, RDF.type, OWL.NamedIndividual))
        g.add((uri, RDF.type, NS.Plant))
        g.add((uri, RDFS.label, Literal(name)))
        added += 1

with open("plant_validation_cache.pkl", "wb") as f:
    pickle.dump(validation_cache, f)

print(f"Plant organisms added: {added}")
save_graph(g, "phytotherapies_augmented.rdf")


## 3. Chemicals from COCONUT + papers via DOI

In [ ]:
# Query COCONUT for all chemical-organism links and load DataFrame
with get_sql_conn() as conn:
    DF = pd.read_sql(
        '''
        SELECT p.molecule_id, p.chemical_class, mo.organism_id, p.molecular_weight,
               p.molecular_formula, m.name AS molecule_name, m.iupac_name,
               e.canonical_smiles
        FROM coconut.properties p
        JOIN coconut.molecule_organism mo ON p.molecule_id = mo.molecule_id
        JOIN coconut.molecules m            ON p.molecule_id = m.id
        JOIN coconut.entries e              ON p.molecule_id = e.molecule_id
        ''', conn).fillna("")

print(f"Total chemical-organism links: {len(DF)}")


In [ ]:
# Enrich ontology with chemical individuals + properties + isDerivedFrom links
set_mw_range_union(g)

added_chems, total_links = set(), 0
for _, row in DF.iterrows():
    mol_id = row["molecule_id"]
    org_id = row["organism_id"]
    chem_uri = make_uri("Chemical", mol_id)
    org_uri  = make_uri("Organism", org_id)

    if chem_uri not in added_chems:
        g.add((chem_uri, RDF.type, OWL.NamedIndividual))
        # Predict phytochemical subclass(es) from SMILES
        smi = row["canonical_smiles"]
        for cls in classify_smiles(smi):
            cls_uri = PHYTOCHEM_CLASSES.get(cls)
            if cls_uri and (cls_uri, RDF.type, OWL.Class) in g:
                g.add((chem_uri, RDF.type, cls_uri))

        if row["molecular_weight"] not in ("", None):
            try:
                g.add((chem_uri, hasMolecularWeight, Literal(float(row["molecular_weight"]))))
            except ValueError:
                pass
        if row["molecular_formula"]:
            g.add((chem_uri, hasMolecularFormula, Literal(row["molecular_formula"])))
        if row["molecule_name"]:
            g.add((chem_uri, hasCommonName, Literal(row["molecule_name"])))
        if row["iupac_name"]:
            g.add((chem_uri, hasIUPACName, Literal(row["iupac_name"])))
        if smi:
            g.add((chem_uri, hasSMILES, Literal(smi)))
            g.add((chem_uri, RDFS.label, Literal(smi)))
        added_chems.add(chem_uri)

    g.add((chem_uri, isDerivedFrom, org_uri))
    total_links += 1

print(f"Unique chemicals added: {len(added_chems)}")
print(f"Chemical-organism links: {total_links}")


In [ ]:
# Link chemicals & organisms to scientific papers via DOI
RESEARCH_ROOT = find_class_by_label_or_local(g, ["ResearchConcept", "Research Concept"])
PAPER_CLASS   = (find_class_by_label_or_local(g, ["ScientificPaper", "Paper"])
                 or NS.ScientificPaper)
if (PAPER_CLASS, RDF.type, OWL.Class) not in g:
    g.add((PAPER_CLASS, RDF.type, OWL.Class))
    g.add((PAPER_CLASS, RDFS.label, Literal("ScientificPaper")))
if RESEARCH_ROOT and (PAPER_CLASS, RDFS.subClassOf, RESEARCH_ROOT) not in g:
    g.add((PAPER_CLASS, RDFS.subClassOf, RESEARCH_ROOT))

if (hasPaper, RDF.type, OWL.ObjectProperty) not in g:
    g.add((hasPaper, RDF.type, OWL.ObjectProperty))
    g.add((hasPaper, RDFS.label, Literal("hasPaper")))
for o in list(g.objects(hasPaper, RDFS.range)):
    g.remove((hasPaper, RDFS.range, o))
g.add((hasPaper, RDFS.range, PAPER_CLASS))

# Pull DOIs from whatever CSV is available
DOI_CSV = next((p for p in [
    "activity_chem_taxon_dosage_refs_with_doi.csv",
    "properties.csv", "citations.csv",
] if Path(p).exists()), None)

papers_added = chem_links = plant_links = 0
if DOI_CSV:
    doi_df = pd.read_csv(DOI_CSV, dtype=str).fillna("")

    def pick(df, opts):
        for c in opts:
            if c in df.columns: return c
        return None

    COL_DOI  = pick(doi_df, ["DOI", "doi", "paper_doi"])
    COL_CHEM = pick(doi_df, ["CHEMID", "molecule_id", "chem_id", "ID"])
    COL_ORG  = pick(doi_df, ["TAXON", "organism_id", "Organism_id", "GENUS_SPECIES"])

    if COL_DOI:
        doi_to_uri = {}
        for _, row in doi_df.iterrows():
            doi = normalize_doi(row[COL_DOI])
            if not doi:
                continue
            paper_uri = doi_to_uri.get(doi) or make_uri("Paper", doi)
            if doi not in doi_to_uri:
                g.add((paper_uri, RDF.type, OWL.NamedIndividual))
                g.add((paper_uri, RDF.type, PAPER_CLASS))
                g.add((paper_uri, RDFS.label, Literal(doi)))
                g.add((paper_uri, hasDOI, Literal(doi)))
                doi_to_uri[doi] = paper_uri
                papers_added += 1

            if COL_CHEM and row[COL_CHEM].strip():
                chem_uri = make_uri("Chemical", row[COL_CHEM])
                if any(g.triples((chem_uri, None, None))):
                    g.add((chem_uri, hasPaper, paper_uri)); chem_links += 1
            if COL_ORG and row[COL_ORG].strip():
                org_uri = make_uri("Organism", row[COL_ORG])
                if any(g.triples((org_uri, None, None))):
                    g.add((org_uri, hasPaper, paper_uri)); plant_links += 1

    print(f"Papers added: {papers_added} | chem→paper: {chem_links} | org→paper: {plant_links}")
else:
    print("No DOI CSV found yet — will be filled in by Dr. Duke's merge later.")

save_graph(g, "phytotherapies_augmented_with_chemicals.rdf")


## 4. CMAUP plant–ingredient–target data

In [ ]:
# Build the CMAUP CSV pipeline (replaces step1.csv → step9.csv chain)
def build_cmaup_csv():
    associations = pd.read_csv(
        "CMAUPv2.0_download_Plant_Ingredient_Associations_allIngredients.txt",
        sep="\t", header=None, names=["Plant_ID", "Ingredient_ID"],
    )
    plants = pd.read_csv("CMAUPv2.0_download_Plants.txt", sep="\t")
    ingr_target = pd.read_csv(
        "CMAUPv2.0_download_Ingredient_Target_Associations_ActivityValues_References.txt",
        sep="\t",
    )
    ingredients = pd.read_csv("CMAUPv2.0_download_Ingredients_All (1).txt", sep="\t")\
        .rename(columns={"np_id": "Ingredient_ID"})
    targets = pd.read_csv("CMAUPv2.0_download_Targets.txt", sep="\t", dtype=str, low_memory=False)

    df = (
        associations
        .merge(plants[["Plant_ID", "Plant_Name"]], on="Plant_ID", how="left")
        .merge(ingr_target[["Ingredient_ID", "Target_ID"]], on="Ingredient_ID", how="left")
        .merge(ingredients[["Ingredient_ID", "pref_name", "MW", "SMILES", "iupac_name"]],
               on="Ingredient_ID", how="left")
    ).drop_duplicates()

    # Resolve Target_ID → Protein_Name and rename column
    targets.columns = [c.strip() for c in targets.columns]
    tmap = dict(zip(targets["Target_ID"].astype(str).str.strip(),
                    targets["Protein_Name"].astype(str).str.strip()))
    df["TargetedPathway"] = df["Target_ID"].astype(str).str.strip().map(tmap).fillna(df["Target_ID"])
    df = df.drop(columns=["Target_ID"])

    # Molecular formula from SMILES
    df["Molecular_Formula"] = df["SMILES"].apply(
        lambda s: rdMolDescriptors.CalcMolFormula(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(str(s) or "") else None
    )

    # Phytochemical class
    df = df.dropna(subset=["SMILES"]).drop_duplicates(subset="SMILES")
    df["SMILES"] = df["SMILES"].astype(str)
    df["Phytochemical_Class"] = df["SMILES"].apply(lambda s: ";".join(classify_smiles(s)))

    df = df.drop(columns=["Plant_ID", "Ingredient_ID"], errors="ignore").replace("n.a.", "")
    df.to_csv("cmaup_clean.csv", index=False)
    return df

cmaup_df = build_cmaup_csv()
print(f"CMAUP rows: {len(cmaup_df)}")


In [ ]:
# Load CMAUP into the ontology
cmaup_df = pd.read_csv("cmaup_clean.csv", na_values=["", "nan", "NaN"])\
    .dropna(subset=["SMILES", "Phytochemical_Class", "Plant_Name"])
for col in ["SMILES", "Phytochemical_Class", "Plant_Name", "Molecular_Formula", "iupac_name", "pref_name"]:
    cmaup_df[col] = cmaup_df[col].astype(str).str.strip()
cmaup_df["MW"] = pd.to_numeric(cmaup_df["MW"], errors="coerce")
cmaup_df = cmaup_df.drop_duplicates(subset=["SMILES"])

# Add plants
existing_plant_labels = {str(o).strip().lower() for _, _, o in g.triples((None, RDFS.label, None))
                        if any(g.triples((_, RDF.type, NS.Plant)))}
plants_added = 0
for name in cmaup_df["Plant_Name"].drop_duplicates():
    if name.lower() in existing_plant_labels:
        continue
    uri = NS[f"Plant_{name.replace(' ', '_')}"]
    g.add((uri, RDF.type, OWL.NamedIndividual))
    g.add((uri, RDF.type, NS.Plant))
    g.add((uri, RDFS.label, Literal(name)))
    plants_added += 1

# Add chemicals (deduped by SMILES)
existing_smiles = {str(o).strip() for _, _, o in g.triples((None, hasSMILES, None))}
chems_added = 0
for _, row in cmaup_df.iterrows():
    if row["SMILES"] in existing_smiles:
        continue
    uri = NS[f"Chemical_{abs(hash(row['SMILES']))}"]
    g.add((uri, RDF.type, OWL.NamedIndividual))
    for cls in row["Phytochemical_Class"].split(";"):
        cls_uri = PHYTOCHEM_CLASSES.get(cls.strip())
        if cls_uri:
            g.add((uri, RDF.type, cls_uri))
    g.add((uri, hasSMILES, Literal(row["SMILES"])))
    g.add((uri, RDFS.label, Literal(row["SMILES"])))
    if pd.notna(row["MW"]):
        g.add((uri, hasMolecularWeight, Literal(row["MW"])))
    if pd.notna(row["Molecular_Formula"]):
        g.add((uri, hasMolecularFormula, Literal(row["Molecular_Formula"])))
    if pd.notna(row["iupac_name"]):
        g.add((uri, hasIUPACName, Literal(row["iupac_name"])))
    if pd.notna(row["pref_name"]):
        g.add((uri, hasCommonName, Literal(row["pref_name"])))
    existing_smiles.add(row["SMILES"])
    chems_added += 1

# isDerivedFrom links
links_added = 0
existing_links = {(str(s), str(o)) for s, _, o in g.triples((None, isDerivedFrom, None))}
for _, row in cmaup_df.iterrows():
    chem_uri = NS[f"Chemical_{abs(hash(row['SMILES']))}"]
    plant_uri = NS[f"Plant_{row['Plant_Name'].replace(' ', '_')}"]
    if (str(chem_uri), str(plant_uri)) not in existing_links:
        g.add((chem_uri, isDerivedFrom, plant_uri))
        existing_links.add((str(chem_uri), str(plant_uri)))
        links_added += 1

print(f"CMAUP: +{plants_added} plants, +{chems_added} chemicals, +{links_added} isDerivedFrom links")
save_graph(g, "phytotherapies_augmented_COCONUT_CMAUP.rdf")


## 5. CMAUP clinical trials

In [ ]:
# Ensure ClinicalTrial schema is aligned under ResearchConcept
TRIAL_CLASS = (find_class_by_label_or_local(g, ["ClinicalTrial", "Clinical Trial"])
               or NS.ClinicalTrial)
RESEARCH_ROOT = find_class_by_label_or_local(g, ["ResearchConcept", "Research Concept"])
if (TRIAL_CLASS, RDF.type, OWL.Class) not in g:
    g.add((TRIAL_CLASS, RDF.type, OWL.Class))
    g.add((TRIAL_CLASS, RDFS.label, Literal("ClinicalTrial")))
if RESEARCH_ROOT and (TRIAL_CLASS, RDFS.subClassOf, RESEARCH_ROOT) not in g:
    g.add((TRIAL_CLASS, RDFS.subClassOf, RESEARCH_ROOT))
for prop, lbl in [(hasClinicalStudy, "hasClinicalStudy"),
                  (hasEvidenceText,  "hasEvidenceText"),
                  (hasNCTId,         "hasNCTId")]:
    if (prop, RDF.type, OWL.ObjectProperty) not in g and (prop, RDF.type, OWL.DatatypeProperty) not in g:
        ptype = OWL.ObjectProperty if prop is hasClinicalStudy else OWL.DatatypeProperty
        g.add((prop, RDF.type, ptype)); g.add((prop, RDFS.label, Literal(lbl)))
for o in list(g.objects(hasClinicalStudy, RDFS.range)):
    g.remove((hasClinicalStudy, RDFS.range, o))
g.add((hasClinicalStudy, RDFS.range, TRIAL_CLASS))


# Load CMAUP PHDA
phda_path = next((p for p in [
    "CMAUPv2.0_download_Plant_Human_Disease_Associations.txt",
    "CMAUPv2.0_download_Plant_Human_Disease_Associations.tsv",
] if Path(p).exists()), None)
if phda_path is None:
    raise FileNotFoundError("CMAUP PHDA file not found")

phda = pd.read_csv(phda_path, sep="\t", dtype=str).fillna("")
plants_df = pd.read_csv("CMAUPv2.0_download_Plants.txt", sep="\t", dtype=str).fillna("")
pid_to_name = dict(zip(plants_df["Plant_ID"], plants_df["Plant_Name"]))

SPLIT_RE = re.compile(r"\s*(?:\|\||\||;;|;|\/\/|,|\n|\r)\s*")

def split_multi(val):
    if not val or str(val).lower() in {"na", "n/a", "none"}:
        return []
    seen, out = set(), []
    for p in SPLIT_RE.split(str(val).strip()):
        p = p.strip()
        if p and p not in seen:
            seen.add(p); out.append(p)
    return out


def mint_trial(plant_id, ev_text):
    ncts = sorted(set(re.findall(r"NCT\d{8}", ev_text)))
    local = f"CT_{ncts[0]}" if ncts else (
        f"CT_p{sanitize_for_uri(plant_id)}_{hashlib.sha1(f'{plant_id}|{ev_text}'.encode()).hexdigest()[:12]}"
    )
    uri = NS[f"ClinicalTrial_{local}"]
    if (uri, RDF.type, TRIAL_CLASS) not in g:
        g.add((uri, RDF.type, OWL.NamedIndividual))
        g.add((uri, RDF.type, TRIAL_CLASS))
        label = ncts[0] if ncts else (ev_text[:120] + ("…" if len(ev_text) > 120 else ""))
        g.add((uri, RDFS.label, Literal(label)))
        g.add((uri, hasEvidenceText, Literal(ev_text, datatype=XSD.string)))
        for n in ncts:
            g.add((uri, hasNCTId, Literal(n)))
    return uri


# Pre-compute plant→chemicals lookup for ingredient-level trials
plant_key_to_chems = defaultdict(set)
for chem_uri, _, plant_uri in g.triples((None, isDerivedFrom, None)):
    pl_local = str(plant_uri).rsplit("#", 1)[-1]
    if pl_local.startswith("Plant_"):
        plant_key_to_chems[pl_local[len("Plant_"):]].add(chem_uri)


trials = plant_links = chem_links = 0
for _, row in phda.iterrows():
    pid = str(row["Plant_ID"]).strip()
    name = pid_to_name.get(pid)
    if not name:
        continue
    p_uri = NS[f"Plant_{sanitize_for_uri(name)}"]
    if not any(g.triples((p_uri, None, None))):
        continue
    key = sanitize_for_uri(name)
    for ev in split_multi(row.get("Association_by_Clinical_Trials_of_Plant", "")):
        t = mint_trial(pid, ev)
        g.add((p_uri, hasClinicalStudy, t)); trials += 1; plant_links += 1
    for ev in split_multi(row.get("Association_by_Clinical_Trials_of_Plant_Ingredients", "")):
        t = mint_trial(pid, ev)
        g.add((p_uri, hasClinicalStudy, t)); trials += 1; plant_links += 1
        for chem in plant_key_to_chems.get(key, []):
            g.add((chem, hasClinicalStudy, t)); chem_links += 1

print(f"Clinical trials: +{trials} | plant→trial: {plant_links} | chem→trial: {chem_links}")
save_graph(g, "phytotherapies_augmented_COCONUT_CMAUP.rdf")


## 6. MACCS fingerprints

In [ ]:
chem_classes = set(PHYTOCHEM_CLASSES.values())
added_maccs = 0
for s, _, o in g.triples((None, RDF.type, None)):
    if o not in chem_classes:
        continue
    if (s, hasMACCs, None) in g:
        continue
    smi = next((str(x) for _, _, x in g.triples((s, hasSMILES, None))), "").strip()
    if not smi:
        continue
    mol = Chem.MolFromSmiles(smi)
    if mol:
        g.add((s, hasMACCs, Literal(MACCSkeys.GenMACCSKeys(mol).ToBitString())))
        added_maccs += 1

print(f"MACCS keys added for {added_maccs} chemicals.")
save_graph(g, "phytotherapies_COCONUT_CMAUP_MACCS.rdf")


## 7. DrugCentral enrichment (therapeutic effects, pathways, ATC)

In [ ]:
# Add InChIKey to chemicals that don't have one
for prop, ptype, lbl in [
    (hasInChIKey,         OWL.DatatypeProperty, "hasInChIKey"),
    (hasTherapeuticEffect,OWL.ObjectProperty,   "hasTherapeuticEffect"),
    (targetsPathway,      OWL.ObjectProperty,   "targetsPathway"),
    (hasValue,            OWL.DatatypeProperty, "hasValue"),
    (hasATC,              OWL.DatatypeProperty, "hasATC"),
]:
    if (prop, RDF.type, ptype) not in g:
        g.add((prop, RDF.type, ptype)); g.add((prop, RDFS.label, Literal(lbl)))

if HAS_INCHI:
    for s, _, smi in g.triples((None, hasSMILES, None)):
        if (s, hasInChIKey, None) in g:
            continue
        mol = Chem.MolFromSmiles(str(smi))
        if mol:
            try:
                ik = RDInchi.MolToInchiKey(mol)
                if ik:
                    g.add((s, hasInChIKey, Literal(ik)))
            except Exception:
                pass

inchikey_to_subj = {str(o): s for s, _, o in g.triples((None, hasInChIKey, None))}
inchis = list(inchikey_to_subj.keys())
print(f"Chemicals with InChIKey: {len(inchis)}")


In [ ]:
# Query DrugCentral: action_type, target name, and ATC code
TherapeuticEffectCls = find_class_by_label_or_local(g, ["TherapeuticEffect"]) or NS.TherapeuticEffect
PathwayClass         = find_class_by_label_or_local(g, ["Pathway"])
if PathwayClass is None:
    raise RuntimeError("Pathway class not found in ontology — required for target linking.")

action_df = pd.DataFrame(); target_df = pd.DataFrame(); atc_df = pd.DataFrame()
if inchis:
    try:
        with get_drugcentral_conn() as conn:
            action_df = pd.read_sql('''
                SELECT s.inchikey, atf.action_type FROM structures s
                JOIN act_table_full atf ON s.id = atf.struct_id
                WHERE s.inchikey = ANY(%s) AND atf.action_type IS NOT NULL
            ''', conn, params=(inchis,))
            target_df = pd.read_sql('''
                SELECT s.inchikey, td.name AS target FROM structures s
                JOIN act_table_full atf ON s.id = atf.struct_id
                JOIN target_dictionary td ON atf.target_id = td.id
                WHERE s.inchikey = ANY(%s)
            ''', conn, params=(inchis,))
            atc_df = pd.read_sql('''
                SELECT s.inchikey, a.code AS atc_code FROM structures s
                JOIN struct2atc s2a ON s.id = s2a.struct_id
                JOIN atc a ON s2a.atc_code = a.code
                WHERE s.inchikey = ANY(%s)
            ''', conn, params=(inchis,))
    except Exception as e:
        print(f"[WARN] DrugCentral query failed: {e}")

print(f"Action rows: {len(action_df)} | Target rows: {len(target_df)} | ATC rows: {len(atc_df)}")


In [ ]:
# Link to TherapeuticEffect and Pathway individuals
effect_cache, pathway_cache = {}, {}

def ensure_effect(label: str) -> URIRef:
    if label in effect_cache: return effect_cache[label]
    uri = NS[f"TherapeuticEffect_{sanitize_for_uri(label)}"]
    if not any(g.triples((uri, None, None))):
        g.add((uri, RDF.type, OWL.NamedIndividual))
        g.add((uri, RDF.type, TherapeuticEffectCls))
        g.add((uri, RDFS.label, Literal(label)))
        g.add((uri, hasValue, Literal(label, datatype=XSD.string)))
    effect_cache[label] = uri
    return uri

def ensure_pathway(label: str) -> URIRef:
    if label in pathway_cache: return pathway_cache[label]
    uri = NS[f"Pathway_{sanitize_for_uri(label)}"]
    if not any(g.triples((uri, None, None))):
        g.add((uri, RDF.type, OWL.NamedIndividual))
        g.add((uri, RDF.type, PathwayClass))
        g.add((uri, RDFS.label, Literal(label)))
        g.add((uri, hasValue, Literal(label, datatype=XSD.string)))
    pathway_cache[label] = uri
    return uri

eff_links = tgt_links = atc_links = 0
for _, row in action_df.iterrows():
    s = inchikey_to_subj.get(row["inchikey"]); a = str(row["action_type"]).strip()
    if s and a:
        g.add((s, hasTherapeuticEffect, ensure_effect(a))); eff_links += 1

for _, row in target_df.iterrows():
    s = inchikey_to_subj.get(row["inchikey"]); t = str(row["target"]).strip()
    if s and t:
        g.add((s, targetsPathway, ensure_pathway(t))); tgt_links += 1

for _, row in atc_df.iterrows():
    s = inchikey_to_subj.get(row["inchikey"]); c = row["atc_code"]
    if s and c:
        g.add((s, hasATC, Literal(c))); atc_links += 1

print(f"DrugCentral: {eff_links} effects, {tgt_links} targets, {atc_links} ATC codes")
save_graph(g, "phytotherapies_enriched_with_actiontype_targets_ATC.rdf")


## 8. ChEMBL drug mechanisms

In [ ]:
# Match ChEMBL drug mechanisms to ontology chemicals via canonical SMILES
mech_csv = "ChEMBL_drug_mechanisms.csv"
try:
    mech_df = pd.read_csv(mech_csv, sep=";")
    if mech_df.shape[1] == 1:
        mech_df = pd.read_csv(mech_csv)
except Exception:
    mech_df = pd.read_csv(mech_csv)
mech_df = mech_df.fillna("")

def pick(df, opts):
    return next((c for c in opts if c in df.columns), None)

COL_SMI   = pick(mech_df, ["Smiles", "SMILES", "smiles", "Canonical_smiles"])
COL_TNAME = pick(mech_df, ["Target Name", "TargetName", "Target"])
COL_ATYPE = pick(mech_df, ["Action Type", "ActionType", "Action"])
if not COL_SMI:
    raise KeyError("No SMILES column in ChEMBL mechanisms CSV")
mech_df[COL_SMI] = mech_df[COL_SMI].astype(str).map(canon_smiles)

# Build ontology canonical-SMILES → subject map (limited to ChemicalConcept subtree)
CHEM_ROOT = find_class_by_label_or_local(g, ["ChemicalConcept", "Chemical Concept"])
chem_cls  = subclass_closure(g, CHEM_ROOT) if CHEM_ROOT else set()
chem_subs = {s for cls in chem_cls for s in g.subjects(RDF.type, cls)}
ont_can_to_subj = {}
for s in chem_subs:
    for _, _, smi in g.triples((s, hasSMILES, None)):
        c = canon_smiles(str(smi))
        if c:
            ont_can_to_subj.setdefault(c, s)

tp_links = te_links = 0
for _, row in mech_df.iterrows():
    smi = row[COL_SMI]
    if not smi:
        continue
    subj = ont_can_to_subj.get(smi)
    if not subj:
        continue
    if COL_TNAME:
        t = str(row[COL_TNAME]).strip()
        if t and t.lower() != "nan":
            g.add((subj, targetsPathway, ensure_pathway(t))); tp_links += 1
    if COL_ATYPE:
        a = str(row[COL_ATYPE]).strip()
        if a and a.lower() != "nan":
            g.add((subj, hasTherapeuticEffect, ensure_effect(a))); te_links += 1

print(f"ChEMBL: targetsPathway +{tp_links}, hasTherapeuticEffect +{te_links}")


In [ ]:
# Filtered ATC: only add to chemicals that have at least one ChEMBL-derived link
enriched = set(g.subjects(targetsPathway, None)) | set(g.subjects(hasTherapeuticEffect, None))
enriched_ik = {str(o) for s in enriched for _, _, o in g.triples((s, hasInChIKey, None))}

added = 0
if not atc_df.empty:
    for _, row in atc_df.iterrows():
        if row["inchikey"] in enriched_ik:
            subj = inchikey_to_subj.get(row["inchikey"])
            if subj and row["atc_code"]:
                g.add((subj, hasATC, Literal(row["atc_code"]))); added += 1
print(f"Filtered ATC additions: {added}")
save_graph(g, "phytotherapies_final_enriched_with_ChEMBL_mechanisms_PATHWAY.rdf")


## 9. Canonical SMILES + inverse edges

In [ ]:
# Add hasCanonicalSMILES, plant→chemical inverse edges, and plant labels
if (hasCompound, RDF.type, OWL.ObjectProperty) not in g:
    g.add((hasCompound, RDF.type, OWL.ObjectProperty))
    g.add((hasCompound, RDFS.label, Literal("hasCompound")))
if (hasCanonicalSMILES, RDF.type, OWL.DatatypeProperty) not in g:
    g.add((hasCanonicalSMILES, RDF.type, OWL.DatatypeProperty))
    g.add((hasCanonicalSMILES, RDFS.label, Literal("hasCanonicalSMILES")))


inv = canon = label_added = 0
for chem, _, plant in g.triples((None, isDerivedFrom, None)):
    if (plant, hasCompound, chem) not in g:
        g.add((plant, hasCompound, chem)); inv += 1
    for _, _, smi in g.triples((chem, hasSMILES, None)):
        c = canon_smiles(str(smi))
        if c and (chem, hasCanonicalSMILES, Literal(c)) not in g:
            g.add((chem, hasCanonicalSMILES, Literal(c))); canon += 1
    if not any(g.triples((plant, RDFS.label, None))):
        genus = next((str(o).strip() for _, _, o in g.triples((plant, hasGenus, None))), "")
        species = next((str(o).strip() for _, _, o in g.triples((plant, hasSpecies, None))), "")
        if genus or species:
            g.add((plant, RDFS.label, Literal(f"{genus} {species}".strip())))
            label_added += 1

print(f"hasCompound: +{inv}, hasCanonicalSMILES: +{canon}, plant labels: +{label_added}")


## 10. Dr. Duke's ethnobotanical merge + DOI extraction

In [ ]:
# Merge ACTIVITIES / AGGREGAC / CHEMICALS / DOSAGE / ETHNOBOT / REFERENCES into one CSV
def read_csv_any(path):
    try:
        df = pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="latin-1", low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    return df

def norm_act(s):  return s.astype(str).str.strip().str.lower()
def norm_chem(s): return s.astype(str).str.strip()

if all(Path(p).exists() for p in
       ["AGGREGAC.csv","ACTIVITIES.csv","CHEMICALS.csv","DOSAGE.csv","ETHNOBOT.csv","REFERENCES.csv"]):
    AGG = read_csv_any("AGGREGAC.csv")
    ACT = read_csv_any("ACTIVITIES.csv")
    CHEM = read_csv_any("CHEMICALS.csv")
    DOS = read_csv_any("DOSAGE.csv")
    ETH = read_csv_any("ETHNOBOT.csv")
    REF = read_csv_any("REFERENCES.csv")

    AGG["chem_key"] = norm_chem(AGG["CHEM"])
    AGG["act_key"]  = norm_act(AGG["ACTIVITY"])
    ACT["act_key"]  = norm_act(ACT["ACTIVITY"])
    CHEM["chem_key"] = norm_chem(CHEM["CHEM"])
    DOS["chem_key"]  = norm_chem(DOS.get("CHEM", pd.Series(dtype=str)))
    ETH["act_key"]   = norm_act(ETH.get("ACTIVITY", pd.Series(dtype=str)))

    ref_map = REF.dropna(subset=["REFERENCE"]).drop_duplicates()\
                 .set_index("REFERENCE")["LONGREF"].to_dict()

    bb = AGG[["chem_key","act_key","CHEM","ACTIVITY","REFERENCE"]].rename(columns={"REFERENCE":"agg_ref"})
    bb = bb.merge(ACT[["act_key","ACTIVITY"]].drop_duplicates(), on="act_key", how="left",
                  suffixes=("","_A"))
    bb["ACTIVITY"] = bb["ACTIVITY_A"].where(bb["ACTIVITY_A"].notna() & (bb["ACTIVITY_A"] != ""), bb["ACTIVITY"])
    bb = bb.drop(columns=[c for c in bb.columns if c.endswith("_A")])
    bb = bb.merge(CHEM[["chem_key","CHEMID"]].drop_duplicates(), on="chem_key", how="left")
    bb = bb.merge(DOS[["chem_key","DOSAGE","REFERENCE"]].drop_duplicates(), on="chem_key", how="left")\
           .rename(columns={"REFERENCE":"dos_ref"})

    merged = bb.merge(ETH[["act_key","GENUS","SPECIES","FAMILY","COUNTRY","TAXON","REFERENCE"]].drop_duplicates(),
                      on="act_key", how="left")

    def collect_longref(row):
        keys = []
        for k in ["agg_ref","dos_ref","REFERENCE"]:
            v = str(row.get(k, "")).strip()
            if v:
                for t in v.replace("|",";").replace(",",";").split(";"):
                    t = t.strip()
                    if t: keys.append(t)
        keys = sorted(set(keys))
        longs = [ref_map.get(k, "") for k in keys if ref_map.get(k, "")]
        return "; ".join(sorted(set(longs))) if longs else "; ".join(keys)

    merged["LONGREF"] = merged.apply(collect_longref, axis=1)
    out = merged[["ACTIVITY","CHEM","CHEMID","DOSAGE","GENUS","SPECIES","FAMILY",
                  "COUNTRY","TAXON","LONGREF"]].fillna("").drop_duplicates()
    out.to_csv("activity_chem_taxon_dosage_refs.csv", index=False)
    print(f"Dr. Duke's merge: {len(out)} rows")


# Extract DOIs from LONGREF
DOI_RE = re.compile(r"(10\.\d{4,9}/[-._;()/:A-Z0-9]+)", re.I)
in_csv  = "activity_chem_taxon_dosage_refs.csv"
out_csv = "activity_chem_taxon_dosage_refs_with_doi.csv"
if Path(in_csv).exists():
    df = pd.read_csv(in_csv, dtype=str).fillna("")
    df["DOI"] = df["LONGREF"].apply(lambda s: (DOI_RE.search(str(s)).group(1) if DOI_RE.search(str(s)) else ""))
    df["DOI"] = df["DOI"].apply(normalize_doi)
    df.to_csv(out_csv, index=False)
    print(f"DOIs extracted → {out_csv}")


## 11. Cleanup pass

In [ ]:
# Fix individuals that were mis-typed as Plant but actually carry hasSMILES
ChemicalConcept = find_class_by_label_or_local(g, ["ChemicalConcept","Chemical Concept"]) or NS.ChemicalConcept
Plant = find_class_by_label_or_local(g, ["Plant"]) or NS.Plant
plant_family = {Plant} | subclass_closure(g, Plant)

mistyped = [(s, str(o)) for s, _, o in g.triples((None, hasSMILES, None))
            if any((s, RDF.type, pc) in g for pc in plant_family)]
print(f"Found {len(mistyped)} mis-typed individuals.")

for subj, smi in mistyped:
    if (subj, RDF.type, ChemicalConcept) not in g:
        g.add((subj, RDF.type, ChemicalConcept))
    for lbl in classify_smiles(smi):
        cls = PHYTOCHEM_CLASSES.get(lbl)
        if cls and (cls, RDF.type, OWL.Class) in g:
            g.add((subj, RDF.type, cls))
    for pc in plant_family:
        if (subj, RDF.type, pc) in g:
            g.remove((subj, RDF.type, pc))


In [ ]:
# Retag bare PlantConcept individuals → Plant
PLANT_ROOT = find_class_by_label_or_local(g, ["PlantConcept","Plant Concept"])
PLANT_LEAF = find_class_by_label_or_local(g, ["Plant"])
moved = 0
if PLANT_ROOT and PLANT_LEAF:
    for s in list(g.subjects(RDF.type, PLANT_ROOT)):
        # Skip schema nodes
        if (s, RDF.type, OWL.Class) in g or (s, RDF.type, OWL.ObjectProperty) in g or (s, RDF.type, OWL.DatatypeProperty) in g:
            continue
        g.add((s, RDF.type, PLANT_LEAF))
        g.remove((s, RDF.type, PLANT_ROOT))
        moved += 1
print(f"Retagged {moved} individuals PlantConcept → Plant")


In [ ]:
# Dedupe individuals by InChIKey / canonical SMILES / DOI / label
NA_TOKENS = {"na","n/a","n.a","n.a.","nan","none","null","not available","not_applicable","not applicable"}

def is_na(lit):
    if not isinstance(lit, Literal): return False
    s = str(lit).strip().lower()
    return s in NA_TOKENS or re.sub(r"[^a-z0-9]+", "", s) in {"na","nan","none","null","notavailable","notapplicable"}

# Remove NA-like literals
na_removed = 0
for s, p, o in list(g.triples((None, None, None))):
    if is_na(o):
        g.remove((s, p, o)); na_removed += 1
print(f"NA-like literals removed: {na_removed}")


def merge_nodes(target, dup):
    for _, p, o in list(g.triples((dup, None, None))):
        if (target, p, o) not in g: g.add((target, p, o))
        g.remove((dup, p, o))
    for s, p, _ in list(g.triples((None, None, dup))):
        if (s, p, target) not in g: g.add((s, p, target))
        g.remove((s, p, dup))

def label_of(node):
    return next((str(o) for _, _, o in g.triples((node, RDFS.label, None))), None)

def pick_canonical(uris):
    labeled = [u for u in uris if label_of(u)]
    return min(labeled or uris, key=lambda u: (len(str(u)), str(u)))

# Bucket chemicals by InChIKey then canonical SMILES then label
buckets = defaultdict(list)
chem_subs = set()
for cls in subclass_closure(g, ChemicalConcept):
    chem_subs |= set(g.subjects(RDF.type, cls))
for subj in chem_subs:
    key = None
    ik = next((str(o) for _, _, o in g.triples((subj, hasInChIKey, None))), None)
    if ik:
        key = ("ikey", ik.strip().lower())
    else:
        smi = next((str(o) for _, _, o in g.triples((subj, hasSMILES, None))), None)
        c = canon_smiles(smi) if smi else None
        if c:
            key = ("smi", c)
        else:
            lbl = label_of(subj)
            if lbl:
                key = ("label", lbl.strip().lower())
    if key:
        buckets[key].append(subj)

merges = 0
for k, uris in buckets.items():
    if len(uris) < 2:
        continue
    target = pick_canonical(uris)
    for dup in set(uris) - {target}:
        merge_nodes(target, dup); merges += 1
print(f"Chemical duplicates merged: {merges}")


In [ ]:
# Inverse-property sync: add (o Q s) for every (s P o) where P owl:inverseOf Q
inv_pairs = {}
for p, q in g.subject_objects(OWL.inverseOf):
    if isinstance(p, URIRef) and isinstance(q, URIRef) and p != q:
        inv_pairs[p] = q; inv_pairs[q] = p

added = 0
for s, p, o in list(g.triples((None, None, None))):
    if isinstance(o, Literal):
        continue
    q = inv_pairs.get(p)
    if q and (o, q, s) not in g:
        g.add((o, q, s)); added += 1

# Symmetric properties
for p in g.subjects(RDF.type, OWL.SymmetricProperty):
    for s, _, o in g.triples((None, p, None)):
        if not isinstance(o, Literal) and (o, p, s) not in g:
            g.add((o, p, s)); added += 1

print(f"Inverse/symmetric triples added: {added}")
save_graph(g, "phytotherapies_inverse_synced.rdf")


## 12. Verification stats

In [ ]:
classes = set(g.subjects(RDF.type, OWL.Class)) | set(g.subjects(RDF.type, RDFS.Class))
obj_props  = set(g.subjects(RDF.type, OWL.ObjectProperty))
data_props = set(g.subjects(RDF.type, OWL.DatatypeProperty))

individuals = set(g.subjects(RDF.type, OWL.NamedIndividual))
for s, _, c in g.triples((None, RDF.type, None)):
    if isinstance(s, URIRef) and c in classes:
        individuals.add(s)
individuals -= classes | obj_props | data_props

instances_by_class = defaultdict(int)
for s, c in g.subject_objects(RDF.type):
    if c not in (OWL.Class, OWL.ObjectProperty, OWL.DatatypeProperty, RDFS.Class):
        instances_by_class[c] += 1

print(f"Triples: {len(g):,}")
print(f"Classes: {len(classes)} | Object props: {len(obj_props)} | Data props: {len(data_props)}")
print(f"Individuals: {len(individuals)}")
print("\nTop instance counts by class:")
for cls, n in sorted(instances_by_class.items(), key=lambda x: -x[1])[:15]:
    lbl = next((str(o) for _, _, o in g.triples((cls, RDFS.label, None))),
               str(cls).rsplit("#", 1)[-1])
    print(f"  {lbl}: {n}")


## 13. ChEBI integration hook (next step)

Drop the `ontology_ext/` package next to this notebook (already built in
`outputs/ontology_ext/`), then download `chebi_lite.owl` from EBI and run:

```
https://ftp.ebi.ac.uk/pub/databases/chebi/ontology/chebi_lite.owl
```


In [ ]:
# === Full ChEBI integration — final, correct version ===
import os, sys, time
from pathlib import Path
from rdflib import Graph, RDF, OWL
from rdflib.namespace import SKOS

sys.path.insert(0, ".")
from ontology_ext import (
    add_alignments, ExternalLinker,
    ChebiSubtreeExtractor, MireotExtractor,
    bind_prefixes,
)
from ontology_ext.namespaces import NCBITAXON, NCIT

os.environ.setdefault("ENTREZ_EMAIL", "ohewryk@sas.upenn.edu")

INPUT = ("phytotherapies_final_enriched_with_ChEMBL_mechanisms_"
         "ATC_filtered_enriched_plants_smiles_fixed_dedup_clean.rdf")
DIST  = Path("dist"); DIST.mkdir(exist_ok=True)
(DIST / "imports").mkdir(exist_ok=True)

# ---- 1. Load the real enriched ontology ----------------------------------
print(f"[1/5] Loading {INPUT} …")
t0 = time.time()
g = Graph(); g.parse(INPUT)
print(f"  Loaded {len(g):,} triples in {time.time()-t0:.1f}s")

# ---- 2. Add TBox alignments ----------------------------------------------
print(f"\n[2/5] Adding TBox alignments …")
report_align = add_alignments(g)
bind_prefixes(g)
print(f"  {report_align}")
g.serialize(DIST / "phytotherapies_aligned.rdf", format="xml")

# ---- 3. External IRI resolution ------------------------------------------
print(f"\n[3/5] Resolving external IRIs (UniChem, NCBI, ATC, DOI) …")
print("       First run hits the network — could take 10-30 min depending on size.")
linker = ExternalLinker(g, ncbi_email=os.environ["ENTREZ_EMAIL"])
linker.link_chemicals_via_inchikey()
linker.link_plants_via_ncbi(existing_cache="ncbi_taxonomy_cache.json")
linker.link_atc_codes()
linker.link_papers_via_doi()
print(f"  Linking report: {linker.report}")
g.serialize(DIST / "phytotherapies_linked.rdf", format="xml")

# ---- 4. Multi-root ChEBI subtree extraction ------------------------------
print(f"\n[4/5] Extracting ChEBI subtrees rooted at your phytochemical classes …")
phyto_to_chebi = {
    "Polyphenol":     "http://purl.obolibrary.org/obo/CHEBI_26195",
    "Carotenoid":     "http://purl.obolibrary.org/obo/CHEBI_23044",
    "Phytosterol":    "http://purl.obolibrary.org/obo/CHEBI_26125",
    "Saponin":        "http://purl.obolibrary.org/obo/CHEBI_26605",
    "Polysaccharide": "http://purl.obolibrary.org/obo/CHEBI_18154",
    "Isoprenoid":     "http://purl.obolibrary.org/obo/CHEBI_24913",
}
ce = ChebiSubtreeExtractor("chebi_lite.owl")
total_chebi = 0
for name, iri in phyto_to_chebi.items():
    n = ce.extract_subtree(root=iri, out_path=None)
    print(f"  {name:15} → {n:>5} classes")
    total_chebi += n

# Pull a few levels of ancestors so the hierarchy connects upward
ce.add_ancestors_of(list(phyto_to_chebi.values()), levels_up=4)
ce.save(str(DIST / "imports" / "chebi_phytochem_subtrees.ttl"))
print(f"  Total ChEBI classes pulled: {total_chebi}")

# NCBITaxon + NCIT slim modules via OLS (small, fast)
print(f"\n  Building NCBITaxon + NCIT slim modules …")
mext = MireotExtractor(g, out_dir=str(DIST / "imports"), cache_dir="external_cache")
mext._build("ncbitaxon", NCBITAXON, "ncbitaxon_slim.ttl")
mext._build("ncit", NCIT, "ncit_slim.ttl")

# ---- 5. Merge everything into one artifact -------------------------------
print(f"\n[5/5] Merging final artifact …")
merged = Graph(); bind_prefixes(merged)
merged.parse(DIST / "phytotherapies_linked.rdf")
for ttl in (DIST / "imports").glob("*.ttl"):
    merged.parse(ttl, format="turtle")
merged.serialize(DIST / "phytotherapies_merged.rdf", format="xml")
print(f"  Final: {len(merged):,} triples → {DIST / 'phytotherapies_merged.rdf'}")

# ---- Sanity check --------------------------------------------------------
print(f"\n=== SANITY CHECK ===")
chebi_matches = sum(1 for _, _, o in merged.triples((None, SKOS.exactMatch, None))
                    if str(o).startswith("http://purl.obolibrary.org/obo/CHEBI_"))
ncbi_matches = sum(1 for _, _, o in merged.triples((None, SKOS.exactMatch, None))
                   if str(o).startswith("http://purl.obolibrary.org/obo/NCBITaxon_"))
chebi_classes = sum(1 for s in merged.subjects(RDF.type, OWL.Class)
                    if str(s).startswith("http://purl.obolibrary.org/obo/CHEBI_"))
print(f"  Chemicals linked to ChEBI:    {chebi_matches}")
print(f"  Plants linked to NCBITaxon:   {ncbi_matches}")
print(f"  ChEBI classes pulled in:      {chebi_classes}")
print(f"\nDone. Open dist/phytotherapies_merged.rdf in Protégé to inspect.")

## 14. External enrichment — names, cross-references, NP classes

Adds PubChem/ChEBI common names, UniChem cross-references, and NPClassifier
chemical classes to the compounds. Operates on the in-memory graph `g` from the
stages above (or load your latest saved RDF into `g` first to resume). All three
steps are cached/resumable.

In [ ]:
## enrichment setup

# ---- External enrichment: HTTP session, disk cache, indexes over g ----
import os, json, time, urllib.parse, requests
from rdflib.namespace import SKOS

CACHE_DIR = "enrich_cache"; os.makedirs(CACHE_DIR, exist_ok=True)
def _cache(name):
    p = os.path.join(CACHE_DIR, name)
    return p, (json.load(open(p)) if os.path.exists(p) else {})

_session = requests.Session()
def _get(url, **kw):  time.sleep(0.22); return _session.get(url, timeout=25, **kw)   # ~4.5/s
def _post(url, **kw): time.sleep(0.25); return _session.post(url, timeout=25, **kw)

def _norm_org(n):
    parts = "".join(c if (c.isalpha() or c==" ") else " " for c in (n or "")).split()
    return f"{parts[0].lower()} {parts[1].lower()}" if len(parts) >= 2 else (parts[0].lower() if parts else "")

# index existing compounds by InChIKey and plants by normalised name
inchikey_to_subj = {str(o): s for s, _, o in g.triples((None, hasInChIKey, None))}
name_to_plant = {}
for s in g.subjects(RDF.type, NS.Plant):
    for _, _, lbl in g.triples((s, RDFS.label, None)):
        name_to_plant.setdefault(_norm_org(str(lbl)), s)
print(f"{len(inchikey_to_subj)} compounds by InChIKey, {len(name_to_plant)} plants by name")

In [ ]:
# PubChem names + CID
# ---- PubChem: compound names + CID xref (resumable) ----
PUG = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{}/property/Title/JSON"
cpath, namecache = _cache("pubchem_names.json"); added = 0
for ik, subj in tqdm(list(inchikey_to_subj.items()), desc="PubChem"):
    if ik not in namecache:
        try:
            p = _get(PUG.format(urllib.parse.quote(ik))).json()["PropertyTable"]["Properties"][0]
            namecache[ik] = {"name": p.get("Title", ""), "cid": p.get("CID")}
        except Exception:
            namecache[ik] = {"name": "", "cid": None}
        if len(namecache) % 200 == 0: json.dump(namecache, open(cpath, "w"))
    rec = namecache[ik]
    if rec.get("name"):
        g.add((subj, hasCommonName, Literal(rec["name"]))); g.add((subj, RDFS.label, Literal(rec["name"]))); added += 1
    if rec.get("cid"):
        g.add((subj, SKOS.exactMatch, URIRef(f"https://pubchem.ncbi.nlm.nih.gov/compound/{rec['cid']}")))
json.dump(namecache, open(cpath, "w")); print("names added:", added)
save_graph(g, "phytotherapies_enriched_names.rdf")

In [ ]:
# UniChem cross-references
# ---- UniChem: cross-refs to ChEMBL/ChEBI/DrugBank/KEGG/HMDB by InChIKey ----
# VERIFY field names if it fails: https://chembl.gitbook.io/unichem/api
UNICHEM = "https://www.ebi.ac.uk/unichem/api/v1/compounds"
XREF = {
 "chembl":      lambda v: f"https://www.ebi.ac.uk/chembl/compound_report_card/{v}",
 "chebi":       lambda v: f"http://purl.obolibrary.org/obo/CHEBI_{str(v).replace('CHEBI:','')}",
 "drugbank":    lambda v: f"https://go.drugbank.com/drugs/{v}",
 "kegg_ligand": lambda v: f"https://www.kegg.jp/entry/{v}",
 "hmdb":        lambda v: f"https://hmdb.ca/metabolites/{v}",
}
upath, ucache = _cache("unichem.json")
for ik, subj in tqdm(list(inchikey_to_subj.items()), desc="UniChem"):
    if ik not in ucache:
        try:
            data = _post(UNICHEM, json={"type": "inchikey", "compound": ik}).json()
            comps = data.get("compounds") or [{"sources": data.get("sources", [])}]
            srcs = {}
            for c in comps:
                for s in c.get("sources", []):
                    nm  = (s.get("shortName") or s.get("name") or "").lower()
                    cid = s.get("compoundId") or s.get("src_compound_id") or s.get("id")
                    if nm and cid: srcs[nm] = cid
            ucache[ik] = srcs
        except Exception:
            ucache[ik] = {}
        if len(ucache) % 200 == 0: json.dump(ucache, open(upath, "w"))
    for nm, val in ucache[ik].items():
        if nm in XREF: g.add((subj, SKOS.exactMatch, URIRef(XREF[nm](val))))
json.dump(ucache, open(upath, "w"))
save_graph(g, "phytotherapies_enriched_xrefs.rdf")

In [ ]:
# NPClassifier for the Unknown compounds
# ---- NPClassifier: real chemical classes for compounds typed Unknown ----
# VERIFY endpoint/keys if it fails: npclassifier.gnps2.org (was npclassifier.ucsd.edu)
NPC = "https://npclassifier.gnps2.org/classify?smiles={}"
hasNPSuperclass, hasNPClass, hasNPPathway = NS.hasNPClassifierSuperclass, NS.hasNPClassifierClass, NS.hasNPClassifierPathway
for p in (hasNPSuperclass, hasNPClass, hasNPPathway): g.add((p, RDF.type, OWL.DatatypeProperty))
npath, npcache = _cache("npclassifier.json")
for subj in tqdm(list(g.subjects(RDF.type, NS.Unknown)), desc="NPClassifier"):
    smi = next((str(o) for _, _, o in g.triples((subj, hasSMILES, None))), "")
    if not smi: continue
    key = str(subj)
    if key not in npcache:
        try:
            j = _get(NPC.format(urllib.parse.quote(smi))).json()
            npcache[key] = {"super": "; ".join(j.get("superclass_results", []) or []),
                            "cls":   "; ".join(j.get("class_results", []) or []),
                            "path":  "; ".join(j.get("pathway_results", []) or [])}
        except Exception:
            npcache[key] = {}
        if len(npcache) % 100 == 0: json.dump(npcache, open(npath, "w"))
    rec = npcache[key]
    if rec.get("super"): g.add((subj, hasNPSuperclass, Literal(rec["super"])))
    if rec.get("cls"):   g.add((subj, hasNPClass,      Literal(rec["cls"])))
    if rec.get("path"):  g.add((subj, hasNPPathway,    Literal(rec["path"])))
json.dump(npcache, open(npath, "w"))
save_graph(g, "phytotherapies_enriched_npclass.rdf")

## 15. IUPAC names for every compound

Fetches a systematic IUPAC name for each compound by InChIKey (batched by PubChem
CID, with a per-InChIKey fallback) and writes `hasIUPACName`. Cached/resumable.
**Saves `phytotherapies_named.rdf`.**

In [ ]:
import json, os, time, requests
from rdflib import Graph, Namespace, RDF, RDFS, OWL, Literal, URIRef
try: from tqdm import tqdm
except Exception:
    def tqdm(x, **k): return x

# --- safety: define these if a fresh kernel hasn't run Cell 0 ---
if "NS" not in globals():
    NS = Namespace("http://www.semanticweb.org/orestah/ontologies/2024/9/phytotherapies#")
if "save_graph" not in globals():
    def save_graph(g, path): g.serialize(destination=path, format="xml")

# --- load the COCONUT-enriched, cleaned ontology ---
COCONUT_FILE = "phytotherapies_coconut_clean.rdf"     # <-- full path if the kernel's cwd differs
if not os.path.exists(COCONUT_FILE):
    raise FileNotFoundError(f"{COCONUT_FILE} not found (cwd is {os.getcwd()})")
g = Graph(); g.parse(COCONUT_FILE)
print("loaded", len(g), "triples from", COCONUT_FILE)
print("COCONUT-2.0 minted nodes (Chemical_ik_):",
      sum(1 for s in set(g.subjects(NS.hasInChIKey, None)) if "Chemical_ik_" in str(s)))

# InChIKey -> CID from your earlier PubChem run (handles either cache layout)
ik_cid = {}
for p in ["pubchem_names.json","enrich_cache/pubchem_names.json","pubchem_cid.json","enrich_cache/pubchem_cid.json"]:
    if os.path.exists(p):
        for ik,v in json.load(open(p)).items():
            c = v.get("cid") if isinstance(v,dict) else v
            if c: ik_cid.setdefault(ik, c)
all_iks = sorted({str(o).strip() for _,_,o in g.triples((None,NS.hasInChIKey,None)) if str(o).strip()})
iupac = json.load(open("iupac_names.json")) if os.path.exists("iupac_names.json") else {}   # ik -> name (resumable)
print(f"{len(all_iks)} compounds; {len(ik_cid)} have a cached CID; {len(iupac)} already named")
sess = requests.Session()
# A) fast path — batch by CID (~200 per call)
cid_ik = {str(c): ik for ik,c in ik_cid.items() if ik not in iupac}
cids = list(cid_ik)
for i in tqdm(range(0,len(cids),200), desc="IUPAC by CID"):
    try:
        time.sleep(0.22)
        r = sess.post("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/property/IUPACName/JSON",
                      data={"cid": ",".join(cids[i:i+200])}, timeout=45)
        for row in r.json()["PropertyTable"]["Properties"]:
            ik = cid_ik.get(str(row.get("CID")))
            if ik: iupac[ik] = row.get("IUPACName","")
    except Exception: pass
    if i % 2000 == 0: json.dump(iupac, open("iupac_names.json","w"))
json.dump(iupac, open("iupac_names.json","w"))
# B) the rest (no CID) — resolve one at a time by InChIKey
rest = [ik for ik in all_iks if ik not in iupac]
for ik in tqdm(rest, desc="IUPAC by InChIKey"):
    try:
        time.sleep(0.22)
        r = sess.get(f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{ik}/property/IUPACName/JSON", timeout=20)
        iupac[ik] = r.json()["PropertyTable"]["Properties"][0].get("IUPACName","")
    except Exception:
        iupac[ik] = ""
    if len(iupac) % 200 == 0: json.dump(iupac, open("iupac_names.json","w"))
json.dump(iupac, open("iupac_names.json","w"))
# write the names into the ontology
added = 0
for s,_,o in g.triples((None, NS.hasInChIKey, None)):
    nm = iupac.get(str(o).strip())
    if nm and (s, NS.hasIUPACName, Literal(nm)) not in g:
        g.add((s, NS.hasIUPACName, Literal(nm))); added += 1
print("added", added, "IUPAC names to the ontology")
save_graph(g, "phytotherapies_named.rdf")

## 16. COCONUT 2.0 occurrences (SDF)

Self-contained: loads `phytotherapies_named.rdf`, streams the COCONUT 2.0 2D SDF,
adds new compounds/plants and `hasCompound`/`isDerivedFrom` occurrence links
(names + NP classes come straight from the SDF), then sanitizes IRIs.
**Saves `phytotherapies_named_coconut.rdf`.** Set `COCO_SDF` to your SDF path.

In [ ]:
import os, re, json, time
from rdflib import Graph, Namespace, RDF, RDFS, OWL, Literal, URIRef
try: from tqdm import tqdm
except Exception:
    def tqdm(x, **k): return x

# ---------- config ----------
NAMED_FILE = "phytotherapies_named.rdf"
COCO_SDF = "coconut_sdf_2d-06-2026.sdf"   # <-- your SDF
OUT_FILE   = "phytotherapies_named_coconut.rdf"

# ---------- namespace + predicates (safe if Cell 0 didn't run) ----------
if "NS" not in globals():
    NS = Namespace("http://www.semanticweb.org/orestah/ontologies/2024/9/phytotherapies#")
hasCompound, isDerivedFrom = NS.hasCompound, NS.isDerivedFrom
hasSMILES, hasInChIKey = NS.hasSMILES, NS.hasInChIKey
hasMolecularFormula, hasCommonName, hasIUPACName = NS.hasMolecularFormula, NS.hasCommonName, NS.hasIUPACName
hasNPSuper, hasNPClass, hasNPPath = NS.hasNPClassifierSuperclass, NS.hasNPClassifierClass, NS.hasNPClassifierPathway
def save_graph(g, path): g.serialize(destination=path, format="xml")

# ---------- 1. load the named ontology ----------
for f in (NAMED_FILE, COCO_SDF):
    if not os.path.exists(f): raise FileNotFoundError(f"{f} not found (cwd {os.getcwd()})")
g = Graph(); g.parse(NAMED_FILE)
print("loaded", len(g), "triples from", NAMED_FILE)

# ---------- 2. indexes from the current graph ----------
INCHIKEY = re.compile(r'^[A-Z]{14}-[A-Z]{10}-[A-Z]$')
def _na(v): return (v or "").strip().lower() in ("", "nan", "none", "null")
def _norm_org(n):
    p = "".join(c if (c.isalpha() or c==" ") else " " for c in (n or "")).split()
    return f"{p[0].lower()} {p[1].lower()}" if len(p) >= 2 else (p[0].lower() if p else "")
def _safe_local(s):
    s = re.sub(r"[^A-Za-z0-9_.\-]", "_", s.strip()); return re.sub(r"_+","_",s).strip("_") or "x"

inchikey_to_subj = {str(o).strip(): s for s,_,o in g.triples((None, hasInChIKey, None)) if str(o).strip()}
name_to_plant = {}
for s in g.subjects(RDF.type, NS.Plant):
    for _,_,l in g.triples((s, RDFS.label, None)): name_to_plant.setdefault(_norm_org(str(l)), s)
print(f"index: {len(inchikey_to_subj)} compounds, {len(name_to_plant)} plants")

# ---------- 3. mint/reuse helpers ----------
def get_or_make_plant(name):
    key = _norm_org(name)
    if key in name_to_plant: return name_to_plant[key]
    uri = NS["Plant_" + _safe_local(name)]
    g.add((uri, RDF.type, OWL.NamedIndividual)); g.add((uri, RDF.type, NS.Plant))
    g.add((uri, RDFS.label, Literal(name.strip())))
    name_to_plant[key] = uri; return uri

def get_or_make_chemical(smiles, inchikey, formula=None, name=None, iupac=None):
    if inchikey in inchikey_to_subj: return inchikey_to_subj[inchikey]
    uri = NS["Chemical_ik_" + inchikey.replace("-", "_")]
    g.add((uri, RDF.type, OWL.NamedIndividual)); g.add((uri, RDF.type, NS.ChemicalConcept))
    if smiles:  g.add((uri, hasSMILES, Literal(smiles)))
    g.add((uri, hasInChIKey, Literal(inchikey)))
    if formula: g.add((uri, hasMolecularFormula, Literal(formula)))
    if iupac:   g.add((uri, hasIUPACName, Literal(iupac)))
    g.add((uri, RDFS.label, Literal(name or iupac or smiles or inchikey)))
    if name:    g.add((uri, hasCommonName, Literal(name)))
    inchikey_to_subj[inchikey] = uri; return uri

# ---------- 4. stream the SDF ----------
WANT = {"canonical_smiles","standard_inchi_key","molecular_formula","name","iupac_name",
        "organisms","np_classifier_superclass","np_classifier_class","np_classifier_pathway"}
def iter_sdf_props(path, want):
    cur, tag = {}, None
    with open(path, "r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            line = line.rstrip("\n")
            if line.startswith("$$$$"):
                if cur: yield cur
                cur, tag = {}, None
            elif line.startswith(">") and "<" in line:
                m = re.search(r"<([^>]+)>", line); tag = m.group(1) if (m and m.group(1) in want) else None
            elif tag is not None:
                if line.strip() == "": tag = None
                else: cur[tag] = (cur[tag] + " " + line.strip()) if tag in cur else line.strip()
    if cur: yield cur

recs = newc = links = 0
for rec in tqdm(iter_sdf_props(COCO_SDF, WANT), total=738823, desc="COCONUT-SDF"):
    recs += 1
    ik = "" if _na(rec.get("standard_inchi_key")) else rec["standard_inchi_key"].strip()
    if not ik: continue
    orgs = rec.get("organisms", "")
    organisms = [] if _na(orgs) else [o.strip() for o in re.split(r"[|;]", orgs) if not _na(o)]
    is_existing = ik in inchikey_to_subj
    if not is_existing and not organisms: continue
    chem = get_or_make_chemical(
        "" if _na(rec.get("canonical_smiles")) else rec["canonical_smiles"].strip(), ik,
        None if _na(rec.get("molecular_formula")) else rec["molecular_formula"].strip(),
        None if _na(rec.get("name")) else rec["name"].strip(),
        None if _na(rec.get("iupac_name")) else rec["iupac_name"].strip())
    if not is_existing: newc += 1
    if not _na(rec.get("np_classifier_superclass")): g.add((chem, hasNPSuper, Literal(rec["np_classifier_superclass"])))
    if not _na(rec.get("np_classifier_class")):      g.add((chem, hasNPClass, Literal(rec["np_classifier_class"])))
    if not _na(rec.get("np_classifier_pathway")):    g.add((chem, hasNPPath,  Literal(rec["np_classifier_pathway"])))
    for org in organisms:
        plant = get_or_make_plant(org)
        if (plant, hasCompound, chem) not in g:
            g.add((plant, hasCompound, chem)); g.add((chem, isDerivedFrom, plant)); links += 1
print(f"COCONUT: {recs} records, {newc} new compounds, {links} new occurrence links")

# ---------- 5. sanitize any unsafe IRIs (organism names) ----------
NS_STR = str(NS); BAD = re.compile(r"[^A-Za-z0-9_.\-]")
def _fix(u):
    if isinstance(u, URIRef) and str(u).startswith(NS_STR):
        loc = str(u)[len(NS_STR):]
        if BAD.search(loc): return URIRef(NS_STR + (re.sub(r"_+","_", BAD.sub("_", loc)).strip("_") or "x"))
    return u
changed = [(s,p,o) for s,p,o in g if (_fix(s),_fix(p),_fix(o)) != (s,p,o)]
for s,p,o in changed: g.remove((s,p,o)); g.add((_fix(s),_fix(p),_fix(o)))
print("sanitized", len(changed), "triples")

# ---------- 6. save ----------
print("total triples now:", len(g))
save_graph(g, OUT_FILE)
print("saved", OUT_FILE)

## 17. Final cleanup + export

Sanitize any remaining unsafe IRIs and merge duplicate compounds (same InChIKey) /
plants (same label) on the in-memory graph from Stage 16, then save the canonical
ontology. This file is what gets uploaded to Box and used to regenerate the
website's `poppy-ontology-real.js`.

In [ ]:
import re
from collections import defaultdict

def save_graph(g, path):
    g.serialize(destination=path, format="xml")

NS_STR = str(NS)
BAD = re.compile(r"[^A-Za-z0-9_.\-]")
if "g" not in globals() or len(g) == 0:
    g = Graph(); g.parse("/Users/Orestah/Downloads/phytotherapies_enriched_coconut.rdf")
print("triples:", len(g))
# 1) sanitize unsafe IRIs  -> fixes the Protégé "illegal character" error
def safe(u):
    if isinstance(u, URIRef) and str(u).startswith(NS_STR):
        loc = str(u)[len(NS_STR):]
        if BAD.search(loc):
            return URIRef(NS_STR + (re.sub(r"_+", "_", BAD.sub("_", loc)).strip("_") or "x"))
    return u
changed = [(s, p, o) for s, p, o in g if (safe(s), safe(p), safe(o)) != (s, p, o)]
for s, p, o in changed:
    g.remove((s, p, o)); g.add((safe(s), safe(p), safe(o)))
print("sanitized URIs in", len(changed), "triples")
# 2) merge duplicate COMPOUNDS (same InChIKey on >1 node)
by_ik = defaultdict(set)
for s, _, o in g.triples((None, hasInChIKey, None)): by_ik[str(o)].add(s)
def keep_compound(subs):                       # prefer original Chemical_<num> over minted ik nodes
    pref = [s for s in subs if "Chemical_ik_" not in str(s)]
    return min(pref or list(subs), key=lambda u: (len(str(u)), str(u)))
cmerged = 0
for ik, subs in by_ik.items():
    if len(subs) < 2: continue
    keep = keep_compound(subs)
    for d in subs - {keep}:
        for p, o in list(g.predicate_objects(d)): g.add((keep, p, o)); g.remove((d, p, o))
        for s, p in list(g.subject_predicates(d)): g.add((s, p, keep)); g.remove((s, p, d))
        cmerged += 1
# 3) merge duplicate PLANTS (identical label on >1 node) — safe; synonyms keep different labels
by_lbl = defaultdict(set)
for s in g.subjects(RDF.type, NS.Plant):
    for _, _, l in g.triples((s, RDFS.label, None)): by_lbl[str(l).strip().lower()].add(s)
def keep_plant(subs):                          # prefer original Organism_<id> over minted Plant_ nodes
    pref = [s for s in subs if "/Organism_" in str(s) or "#Organism_" in str(s)]
    return min(pref or list(subs), key=lambda u: (len(str(u)), str(u)))
pmerged = 0
for lbl, subs in by_lbl.items():
    if len(subs) < 2: continue
    keep = keep_plant(subs)
    for d in subs - {keep}:
        for p, o in list(g.predicate_objects(d)): g.add((keep, p, o)); g.remove((d, p, o))
        for s, p in list(g.subject_predicates(d)): g.add((s, p, keep)); g.remove((s, p, d))
        pmerged += 1
print(f"merged {cmerged} duplicate compounds, {pmerged} duplicate plants")
save_graph(g, "phytotherapies_coconut_clean.rdf")
print("done -> phytotherapies_coconut_clean.rdf")